<a href="https://colab.research.google.com/github/raimaravi13-netizen/Raima-file/blob/main/titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [41]:
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [42]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df=pd.read_csv('/content/tested.csv')

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
# Remove duplicate records
df = df.drop_duplicates().copy()

# Select useful input features
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

target = "Survived"

X = df[features].copy()
y = df[target].copy()

In [ ]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Sex",
    "Embarked"
]

In [ ]:
# Handle missing numerical values
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Handle missing categorical values and encode them
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


In [ ]:
# Combine preprocessing methods
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

In [ ]:
print("\nInput features (X):")
print(X.head())

print("\nTarget variable (Y):")
print(y.head())

In [ ]:
#  TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:

#  APPLY RANDOM FOREST

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])


In [ ]:
model.fit(X_train, y_train)

In [ ]:
#  MAKE PREDICTIONS

y_pred = model.predict(X_test)

print("\nPredicted values:")
print(y_pred[:20])

print("\nActual values:")
print(y_test.to_numpy()[:20])

In [ ]:
#  EVALUATE THE MODEL

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)



print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")
print("F1 Score:", round(f1 * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
# CONFUSION MATRIX

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Did Not Survive", "Survived"],
    yticklabels=["Did Not Survive", "Survived"]
)


In [ ]:
plt.title("Titanic Survival - Confusion Matrix")
plt.xlabel("Predicted Values")
plt.ylabel("Actual Values")
plt.show()

In [ ]:
# Get feature names after preprocessing
feature_names = (
    model.named_steps["preprocessor"].get_feature_names_out()
)

importance_values = (
    model.named_steps["classifier"].feature_importances_
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

In [ ]:
print(importance_df)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance_df,
    x="Importance",
    y="Feature"
)

In [ ]:
plt.title("Titanic Survival - Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

In [ ]:
cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)


In [ ]:
print("Cross Validation Scores:", cv_scores)

print(
    "Average Accuracy:",
    round(cv_scores.mean() * 100, 2),
    "%"
)

In [ ]:
 #HYPERPARAMETER TUNING



param_grid = {
    "classifier__n_estimators": [100, 150],
    "classifier__max_depth": [5, 10, None],
    "classifier__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print(
    "Best Cross Validation Score:",
    round(grid_search.best_score_ * 100, 2),
    "%"
)

In [ ]:
# EVALUATE TUNED MODEL

best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

print(
    "\nTuned Model Accuracy:",
    round(
        accuracy_score(y_test, tuned_predictions) * 100, 2
    ),
    "%"
)


In [ ]:

# SAVE MODEL

joblib.dump(
    best_model,
    "titanic_random_forest_model.pkl"
)

print("\nModel saved successfully!")

files.download("titanic_random_forest_model.pkl")

In [ ]:
print("Project: Titanic Survival Prediction")
print("Algorithm: Random Forest Classifier")
print("Dataset: Titanic Kaggle Dataset")

print("Total records:", len(df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")
print("F1 Score:", round(f1 * 100, 2), "%")

print("\nAll operations completed successfully!")
